#**[프로젝트] 서울 전월세 가격 변동 요인 분석 - 금리와 지하철 접근성을 중심으로**
- [서울시 부동산 전월세가 정보](https://data.seoul.go.kr/dataList/OA-21276/S/1/datasetView.do) : 서울 부동산 정보광장에서 제공하는 전월세가 정보입니다. 자치구, 법정동, 건물명, 임대건물명, 전월세구분, 보증금, 임대료, 계약년도 등의 전월세 정보를 제공합니다.

# 1.데이터분석의 주제 선정
 - 서울시 전월세 실거래 데이터와 금리 및 지하철 접근성 데이터를 결합하여 금리 변화와 주거 입지에 따른 월세 가격 변동을 분석한다.
 - 이를 통해 월세 가격의 주요 변동 요인과 시기별, 지역별 가격 차이를 파악하고 청년층의 주거지역 및 거주 계획 수립을 지원하는 데이터 기반 정보를 제공한다.


# 2.데이터 수집
## 원본 데이터 출처
- [서울시 부동산 전월세가 정보](https://data.seoul.go.kr/dataList/OA-21276/S/1/datasetView.do)
- [한국은행 기준금리 및 여수신금리](ecos.bok.or.kr/)
- [지하철역 좌표정보](https://t-data.seoul.go.kr/dataprovide/trafficdataviewfile.do?data_id=36)

## 활용 데이터
- 데이터 공유폴더 : https://drive.google.com/drive/folders/1tdWmbPR2a2lcKZbFxpWXeqmJE8CmM3yl?usp=sharing

- Data1 : 서울시 전월세 실거래 데이터 (2017-2025)

- Data2 : 한국은행 기준금리

- Data3 : 지하철역 좌표정보

##2-1. colab에서 한글 사용하기

In [1]:
!pip install koreanize-matplotlib geopy tqdm -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 24.0 MB/s eta 0:00:00
^C


In [2]:
import requests
import time
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
import koreanize_matplotlib
from google.colab import files
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

##2-2. 데이터 불러오기 (공통)

In [ ]:
df_list = []

# 2016-2021 txt 파일 불러오기
for year in range(2016, 2022):
    file_path = (f"/content/rent_{year}.txt")

    df_year = pd.read_csv(file_path, encoding="cp949", low_memory=False)
    df_list.append(df_year)

    print(f"{year}년: {len(df_year)}건")

# 2022~2025 CSV 파일 불러오기
for year in range(2022, 2026):
    file_path = (f"/content/rent_{year}.csv")

    # 2023년만 utf-8, 나머지는 cp949
    if year == 2023:
        encoding = "utf-8-sig"
    else:
        encoding = "cp949"

    df_year = pd.read_csv(file_path, encoding=encoding, low_memory=False)
    df_list.append(df_year)

    print(f"{year}년: {len(df_year)}건")

2016년: 382412건
2017년: 398292건
2018년: 436595건
2019년: 458969건
2020년: 509038건
2021년: 524675건
2022년: 560665건
2023년: 545813건
2024년: 522001건
2025년: 1084942건


In [ ]:
# 10개년 데이터 통합
df = pd.concat(df_list, ignore_index=True)

In [ ]:
df.columns

Index(['접수년도', '자치구코드', '자치구명', '법정동코드', '법정동명', '지번구분코드', '지번구분', '본번', '부번',
       '층', '계약일', '전월세구분', '임대면적', '보증금(만원)', '임대료(만원)', '건물명', '건축년도',
       '건물용도', '계약기간', '신규계약구분', '갱신청구권사용', '종전보증금', '종전임대료'],
      dtype='object')

## 2-3 데이터 확인하기
건수, 결측치, 중복, 이상치를 확인하고 정리

### 2-3-1. 건수
원본 데이터 건수 확인

In [ ]:
df.shape

(5423402, 23)

### 2-3-2. 중복
2025년 데이터가 타 연도 대비 약 2배 많아 중복 데이터 여부 확인

In [ ]:
print(f"전체 행 수: {len(df):,}")

전체 행 수: 5,423,402


In [ ]:
print("중복 행 수:", f"{df.duplicated().sum():,}")

중복 행 수: 452,458


In [ ]:
# 연도별 중복 발생 확인
for i, df_year in enumerate(df_list, start=2016):
    print(
        i,
        f"전체: {len(df_year):,}",
        f"중복: {df_year.duplicated().sum():,}"
    )

2016 전체: 382,412 중복: 0
2017 전체: 398,292 중복: 0
2018 전체: 436,595 중복: 0
2019 전체: 458,969 중복: 0
2020 전체: 509,038 중복: 0
2021 전체: 524,675 중복: 0
2022 전체: 560,665 중복: 0
2023 전체: 545,813 중복: 0
2024 전체: 522,001 중복: 1
2025 전체: 1,084,942 중복: 446,244


In [ ]:
df_2025 = df_list[-1]

# 실제 중복 데이터 확인
display(df_2025[df_2025.duplicated(keep=False)].head(20))

,접수년도,자치구코드,자치구명,법정동코드,법정동명,지번구분코드,지번구분,본번,부번,층,...,보증금(만원),임대료(만원),건물명,건축년도,건물용도,계약기간,신규계약구분,갱신청구권사용,종전보증금,종전임대료
133690,2025,11260.0,중랑구,10500.0,망우동,NaN,NaN,NaN,NaN,NaN,...,500,40,NaN,2015.0,단독다가구,25.09~27.09,신규,NaN,NaN,NaN
133691,2025,11410.0,서대문구,11600.0,창천동,NaN,NaN,NaN,NaN,NaN,...,300,51,NaN,1984.0,단독다가구,25.10~26.04,신규,NaN,NaN,NaN
133692,2025,11590.0,동작구,10700.0,사당동,NaN,NaN,NaN,NaN,NaN,...,500,32,NaN,1996.0,단독다가구,25.10~27.10,신규,NaN,NaN,NaN
133693,2025,11710.0,송파구,10600.0,삼전동,1.0,대지,106.0,10.0,2.0,...,500,55,(106-10),2011.0,연립다세대,25.10~27.10,신규,NaN,NaN,NaN
133694,2025,11710.0,송파구,10600.0,삼전동,1.0,대지,112.0,8.0,2.0,...,12000,0,(112-8),2001.0,연립다세대,25.10~27.10,갱신,NaN,12000.0,NaN
133695,2025,11710.0,송파구,10600.0,삼전동,1.0,대지,116.0,9.0,3.0,...,3000,74,(116-9),2007.0,연립다세대,25.12~27.11,신규,NaN,NaN,NaN
133696,2025,11710.0,송파구,10400.0,송파동,1.0,대지,117.0,30.0,3.0,...,15000,14,(117-30),2022.0,연립다세대,25.10~27.10,신규,NaN,NaN,NaN
133697,2025,11710.0,송파구,10600.0,삼전동,1.0,대지,106.0,10.0,2.0,...,500,55,(106-10),2011.0,연립다세대,25.10~27.10,신규,NaN,NaN,NaN
133698,2025,11230.0,동대문구,10300.0,제기동,1.0,대지,1185.0,0.0,1.0,...,12000,50,(1185-0),1980.0,연립다세대,25.10~27.10,신규,NaN,NaN,NaN
133699,2025,11560.0,영등포구,11700.0,당산동,1.0,대지,121.0,126.0,1.0,...,1000,40,(121-126),1997.0,오피스텔,25.10~27.10,신규,NaN,NaN,NaN


2025년에는 동일한 데이터가 중복 기재된 것으로 판단되어 중복 제거

In [ ]:
# 중복 제거
df = df.drop_duplicates()

In [ ]:
print(f"중복 제거 후 전체 행 수: {len(df):,}")

중복 제거 후 전체 행 수: 4,970,944


### 2-3-3. 필터링
월세 데이터만 필터링하여 데이터 프레임 크기 축소

In [ ]:
# 월세 데이터만 필터링
df = df[df["전월세구분"] == "월세"]

In [ ]:
print(f"월세 데이터만 필터링 후 전체 행 수: {len(df):,}")

월세 데이터만 필터링 후 전체 행 수: 2,382,333


In [ ]:
# 중간 저장
output_path = "/content/rental_filtered.csv"
df.to_csv(output_path,index=False,encoding="utf-8-sig")


files.download("/content/rental_filtered.csv")
print(f"저장 완료: {output_path}")
print(f"데이터 크기: {df.shape}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

저장 완료: /content/rental_filtered.csv
데이터 크기: (2382333, 23)


### 2-3-4. 결측치
결측치 확인

In [ ]:
# 결측치 갯수 확인
df.isnull().sum()

,0
접수년도,0
자치구코드,1
자치구명,1
법정동코드,6
법정동명,7
지번구분코드,896048
지번구분,896048
본번,895436
부번,895436
층,896331


## 2-4 데이터 전처리하기
본번, 부번, 법정동명, 자치구명 컬럼을 이용하여 address 컬럼 파생 후 결측 행 제거

In [ ]:
df.columns

Index(['접수년도', '자치구코드', '자치구명', '법정동코드', '법정동명', '지번구분코드', '지번구분', '본번', '부번',
       '층', '계약일', '전월세구분', '임대면적', '보증금(만원)', '임대료(만원)', '건물명', '건축년도',
       '건물용도', '계약기간', '신규계약구분', '갱신청구권사용', '종전보증금', '종전임대료'],
      dtype='object')

### 2-4-1. address 컬럼 생성

In [ ]:
# 본번 / 부번 / 법정동명 / 자치구명 데이터 타입 확인

print(df["자치구명"].dtype)
print(df["법정동명"].dtype)
print(df["본번"].dtype)
print(df["부번"].dtype)

object
object
float64
float64


In [ ]:
# 주소 생성 규칙 함수
def make_address(row):
    sido = "서울특별시"
    gu = row["자치구명"]
    dong = row["법정동명"]
    bonbun = row["본번"]
    bubun = row["부번"]

    # 필수 주소 정보가 없는 경우
    if (
        pd.isna(gu)
        or pd.isna(dong)
        or pd.isna(bonbun)
    ):
        return pd.NA

    # 문자열 공백 제거 후 빈 값 확인
    gu = str(gu).strip()
    dong = str(dong).strip()

    if gu == "" or dong == "":
        return pd.NA

    # 본번이 숫자가 아닌 경우
    try:
        bonbun = int(float(bonbun))
    except (ValueError, TypeError):
        return pd.NA

    # 부번이 없거나 0인 경우
    if pd.isna(bubun) or bubun == 0:
        return f"{sido} {gu} {dong} {bonbun}"

    # 부번이 존재하는 경우
    try:
        bubun = int(float(bubun))
    except (ValueError, TypeError):
        return f"{sido} {gu} {dong} {bonbun}"

    return f"{sido} {gu} {dong} {bonbun}-{bubun}"

In [ ]:
# 완전한 주소 생성
df["address"] = df.apply(make_address, axis=1)

In [ ]:
# 결과 확인
print(f"주소 생성 완료: {df['address'].notna().sum():,}건")
print(f"주소 결측: {df['address'].isna().sum():,}건")

print(df[["자치구명", "법정동명", "본번", "부번", "address"]].head(10))

주소 생성 완료: 1,486,896건
주소 결측: 895,437건
   자치구명   법정동명   본번   부번             address
0   용산구  원효로2가  1.0  0.0   서울특별시 용산구 원효로2가 1
1   마포구   노고산동  1.0  1.0  서울특별시 마포구 노고산동 1-1
2   마포구   노고산동  1.0  1.0  서울특별시 마포구 노고산동 1-1
3   마포구   노고산동  1.0  1.0  서울특별시 마포구 노고산동 1-1
4   마포구   노고산동  1.0  1.0  서울특별시 마포구 노고산동 1-1
5   마포구   노고산동  1.0  1.0  서울특별시 마포구 노고산동 1-1
7   마포구   노고산동  1.0  1.0  서울특별시 마포구 노고산동 1-1
8   마포구   노고산동  1.0  1.0  서울특별시 마포구 노고산동 1-1
10  마포구   노고산동  1.0  1.0  서울특별시 마포구 노고산동 1-1
12  마포구   노고산동  1.0  1.0  서울특별시 마포구 노고산동 1-1


In [ ]:
# 주소 결측 행 확인
df.loc[df["address"].isna(), ["자치구명", "법정동명", "본번", "부번", "address"]]

,자치구명,법정동명,본번,부번,address
256620,서초구,신원동,NaN,NaN,<NA>
256625,서초구,신원동,NaN,NaN,<NA>
256628,서초구,신원동,NaN,NaN,<NA>
256629,서초구,신원동,NaN,NaN,<NA>
257214,종로구,명륜3가,NaN,NaN,<NA>
...,...,...,...,...,...
5423397,관악구,신림동,NaN,NaN,<NA>
5423398,관악구,봉천동,NaN,NaN,<NA>
5423399,동대문구,휘경동,NaN,NaN,<NA>
5423400,광진구,구의동,NaN,NaN,<NA>


In [ ]:
# 주소 결측 데이터 제거
df = df.dropna(subset=["address"]).copy()
print(f"주소 결측 제거 후: {len(df):,}건")

주소 결측 제거 후: 1,486,896건


In [ ]:
# 고유 주소 건수 확인
print(f"전체 데이터: {len(df):,}건")
print(f"고유 주소: {df['address'].nunique():,}건")

전체 데이터: 1,486,896건
고유 주소: 83,844건


In [ ]:
# 중간 저장
output_path = "/content/rental_filtered_address.csv"
df.to_csv(output_path,index=False,encoding="utf-8-sig")

files.download("/content/rental_filtered_address.csv")
print(f"저장 완료: {output_path}")
print(f"데이터 크기: {df.shape}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

저장 완료: /content/rental_filtered_address.csv
데이터 크기: (1486896, 24)


### 2-4-2. 지오코딩 (대시 보드 제작용)
**전체 흐름**

- **0단계: 대시 보드 제작 용 2025년 데이터만 필터링**
- **1단계: 카카오 로컬 API**: 중복을 제거한 고유 주소만 추출해 좌표 변환<br>
- **2단계: VWorld 주소 API**: 1단계에서 실패한 주소만 재시도<br>
- **3단계: Google Geocoding API**: 2단계에서도 실패한 주소를 마지막으로 재시도

3단계를 모두 거친 최종 좌표를 원본 데이터의 `위도`, `경도` 컬럼에 병합

#### 환경 설정

In [ ]:
# API 키
KAKAO_REST_API_KEY = "입력"
VWORLD_API_KEY = "입력"
GOOGLE_API_KEY = "입력"

# 병렬 요청
MAX_WORKERS = 5
REQUEST_TIMEOUT = 10

In [ ]:
# 데이터 불러오기
df = pd.read_csv(
    "/content/rental_filtered_address.csv",
    encoding="utf-8-sig"
)

In [ ]:
# 2025년 데이터만 필터링
df_2025 = df[df["계약일"].astype(str).str.startswith("2025")]

# 데이터 행수 확인
print(f"2025년 데이터 행수: {len(df_2025):,}")

2025년 데이터 행수: 246,147


#### 1단계 — 카카오 로컬 API 지오코딩

카카오 로컬 API의 **주소 검색**을 우선 사용하고, 결과가 없으면 **키워드 검색**으로 한 번 더 시도합니다(도로명, 건물명이 섞여 인식되지 않는 주소를 보완하기 위함).

In [ ]:
KAKAO_ADDRESS_URL = "https://dapi.kakao.com/v2/local/search/address.json"
KAKAO_KEYWORD_URL = "https://dapi.kakao.com/v2/local/search/keyword.json"
KAKAO_HEADERS = {"Authorization": f"KakaoAK {KAKAO_REST_API_KEY}"}

def geocode_kakao(address):
    """카카오 주소 검색 → 실패 시 키워드 검색으로 재시도. (위도, 경도) 또는 (nan, nan) 반환."""
    try:
        for url in (KAKAO_ADDRESS_URL, KAKAO_KEYWORD_URL):
            response = requests.get(
                url, headers=KAKAO_HEADERS, params={"query": address}, timeout=REQUEST_TIMEOUT
            )
            if response.status_code == 200:
                documents = response.json().get("documents", [])
                if documents:
                    result = documents[0]
                    # 카카오 API: y = 위도, x = 경도
                    return float(result["y"]), float(result["x"])
            elif response.status_code == 429:
                print("ERROR: 카카오 API 요청 제한(429) - 잠시 대기 후 계속")
                time.sleep(1)
        return np.nan, np.nan
    except Exception as e:
        print(f"ERROR: 카카오 API 오류 ({address}) - {e}")
        return np.nan, np.nan

In [ ]:
# 고유 주소 리스트를 병렬로 지오코딩하고 {주소: (위도, 경도)} 딕셔너리를 반환
def run_geocoding(addresses, geocode_func, desc):
    results = {}
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(geocode_func, addr): addr for addr in addresses}
        for future in tqdm(as_completed(futures), total=len(futures), desc=desc, unit="건"):
            addr = futures[future]
            results[addr] = future.result()
    return results

In [ ]:
# 고유 주소만 추출 - 중복 API 호출 방지
unique_addresses = df["address"].dropna().drop_duplicates().tolist()
print(f"전체 행: {len(df):,}건 | 고유 주소: {len(unique_addresses):,}건")

kakao_results = run_geocoding(unique_addresses, geocode_kakao, desc="1단계: 카카오 지오코딩")

kakao_df = pd.DataFrame(
    [{"address": addr, "위도": lat, "경도": lon} for addr, (lat, lon) in kakao_results.items()]
)

kakao_success = kakao_df["위도"].notna().sum()
print(f"카카오 성공: {kakao_success:,}건 / 실패: {len(kakao_df) - kakao_success:,}건")

1단계 결과를 원본 데이터에 병합

이후 단계에서는 **여기서 실패한 주소만** 대상으로 재시도합니다.

In [ ]:
df = df.merge(kakao_df, on="address", how="left")

print(f"전체 데이터: {len(df):,}건")
print(f"위경도 확보: {df['위도'].notna().sum():,}건")
print(f"위경도 미확보: {df['위도'].isna().sum():,}건")

#### 2단계 — VWorld 주소 API로 실패 주소 재시도

카카오에서 실패한 주소 중 **고유 주소만** 다시 추출해 VWorld(국토교통부 제공) 지번 주소 검색 API로 재시도합니다.

In [ ]:
VWORLD_URL = "https://api.vworld.kr/req/address"


def geocode_vworld(address):
    """VWorld 지번(PARCEL) 주소 검색. (위도, 경도) 또는 (nan, nan) 반환."""
    params = {
        "service": "address",
        "request": "getcoord",
        "version": "2.0",
        "crs": "EPSG:4326",   # WGS84 경위도로 반환
        "address": address,
        "refine": "true",
        "simple": "false",
        "format": "json",
        "type": "PARCEL",     # 지번 주소 검색
        "key": VWORLD_API_KEY,
    }

    try:
        response = requests.get(VWORLD_URL, params=params, timeout=REQUEST_TIMEOUT)
        response.raise_for_status()
        data = response.json()

        response_data = data.get("response", {})
        if response_data.get("status") != "OK":
            return np.nan, np.nan

        result = response_data.get("result")
        if isinstance(result, list):
            result = result[0] if result else None
        if not result:
            return np.nan, np.nan

        point = result.get("point")
        if not point:
            return np.nan, np.nan

        # VWorld point: x = 경도, y = 위도
        return float(point["y"]), float(point["x"])

    except Exception as e:
        print(f"⚠️ VWorld API 오류 ({address}): {e}")
        return np.nan, np.nan

In [ ]:
# 1단계에서 실패한 행만 대상으로, 고유 주소만 추출
failed_mask = df["위도"].isna() | df["경도"].isna()
retry_addresses_1 = df.loc[failed_mask, "address"].dropna().drop_duplicates().tolist()

print(f"2단계 재시도 대상 (고유 주소): {len(retry_addresses_1):,}건")

vworld_results = run_geocoding(retry_addresses_1, geocode_vworld, desc="2단계: VWorld 지오코딩")

vworld_df = pd.DataFrame(
    [{"address": addr, "위도_vworld": lat, "경도_vworld": lon} for addr, (lat, lon) in vworld_results.items()]
)

vworld_success = vworld_df["위도_vworld"].notna().sum()
print(f"VWorld 성공: {vworld_success:,}건 / 실패: {len(vworld_df) - vworld_success:,}건")

2단계 결과 병합

카카오 좌표가 없는 행에 한해서만 VWorld 결과로 채웁니다 (`fillna` 사용 — 기존에 이미 확보된 좌표는 덮어쓰지 않음).

In [ ]:
df = df.merge(vworld_df, on="address", how="left")

df["위도"] = df["위도"].fillna(df["위도_vworld"])
df["경도"] = df["경도"].fillna(df["경도_vworld"])

df = df.drop(columns=["위도_vworld", "경도_vworld"])

print(f"전체 데이터: {len(df):,}건")
print(f"위경도 확보: {df['위도'].notna().sum():,}건")
print(f"위경도 미확보: {df['위도'].isna().sum():,}건")

#### 3단계 — Google Geocoding API로 남은 실패 주소 재시도

카카오, VWorld 두 국내 API에서도 인식되지 않은 주소를 Google Geocoding API로 마지막으로 재시도합니다.

In [ ]:
from pathlib import Path

# 2단계까지 완료된 파일을 불러옵니다.

DATA_DIR = Path(r"C:\Users\SBA\Desktop\mypython\final-project\data")
MAIN_FILE = DATA_DIR / "Rental_and_Lease_Data_Geocoded.csv"
df = pd.read_csv(MAIN_FILE)

# Google Geocoding API
GOOGLE_URL = "https://maps.googleapis.com/maps/api/geocode/json"


def geocode_google(address):
    params = {
        "address": address,
        "key": GOOGLE_API_KEY,
        "language": "ko",
        "region": "kr",
    }

    try:
        response = requests.get(
            GOOGLE_URL,
            params=params,
            timeout=REQUEST_TIMEOUT
        )
        response.raise_for_status()
        data = response.json()

        if data.get("status") != "OK":
            return np.nan, np.nan

        results = data.get("results", [])
        if not results:
            return np.nan, np.nan

        location = results[0]["geometry"]["location"]

        return float(location["lat"]), float(location["lng"])

    except Exception as e:
        print(f"ERROR: Google API 오류 ({address}) - {e}")
        return np.nan, np.nan


In [ ]:
# 위도 또는 경도 중 하나라도 없는 행
failed_mask = df["위도"].isna() | df["경도"].isna()

# 실패한 행에서 주소만 추출 → 결측 제거 → 중복 제거
retry_addresses_2 = (
    df.loc[failed_mask, "address"]
    .dropna()
    .drop_duplicates()
    .tolist()
)

print(f"3단계 재시도 대상 (고유 주소): {len(retry_addresses_2):,}건")


# Google Geocoding API 실행
google_results = run_geocoding(
    retry_addresses_2,
    geocode_google,
    desc="3단계: Google 지오코딩"
)

# Google 결과를 DataFrame으로 변환
google_df = pd.DataFrame(
    [
        {
            "address": addr,
            "위도_google": lat,
            "경도_google": lon
        }
        for addr, (lat, lon) in google_results.items()
    ],
    columns=["address", "위도_google", "경도_google"]
)

# 성공 / 실패 건수
google_success = google_df["위도_google"].notna().sum()

print(
    f"Google 성공: {google_success:,}건 / "
    f"실패: {len(google_df) - google_success:,}건"
)

### 2-4-3. 결과 병합

In [ ]:
df = df.merge(google_df, on="address", how="left")

df["위도"] = df["위도"].fillna(df["위도_google"])
df["경도"] = df["경도"].fillna(df["경도_google"])

df = df.drop(columns=["위도_google", "경도_google"])

print(f"전체 데이터: {len(df):,}건")
print(f"위경도 확보: {df['위도'].notna().sum():,}건")
print(f"위경도 미확보: {df['위도'].isna().sum():,}건")

In [ ]:
# 최종적으로도 좌표를 찾지 못한 주소 (수동 확인용)
final_failed = df[df["위도"].isna() | df["경도"].isna()]

print(f"최종 미확보: {final_failed['address'].nunique():,}건 (고유 주소 기준)")

final_failed[["자치구명", "법정동명", "지번구분", "본번", "부번", "address"]].drop_duplicates(
    subset=["address"]
).to_csv("지오코딩_최종실패주소.csv", index=False, encoding="utf-8-sig")

print("저장 완료: 지오코딩_최종실패주소.csv")

In [ ]:
# 대시 보드 제작용 데이터 저장
OUTPUT_PATH = "df_2025_geo.csv"
df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print(f"저장 완료: {OUTPUT_PATH}")
print(f"데이터 크기: {df.shape}")

### 2-4-4. 금리와 상관관계 분석을 위한 데이터 정제

In [ ]:
df = pd.read_csv("/content/rental_filtered_address.csv", encoding = "utf-8-sig")

/tmp/ipykernel_14694/1051325801.py:1: DtypeWarning: Columns (18,19,20) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/content/rental_filtered_address.csv", encoding = "utf-8-sig")


In [ ]:
# 10㎡ 미만 거래 제외
df = df[df["임대면적"] >= 10].copy()

print(f"10㎡ 미만 제거 후: {len(df):,}건")
print(f"임대면적 최소값: {df['임대면적'].min():.2f}㎡")

10㎡ 미만 제거 후: 1,486,597건
임대면적 최소값: 10.00㎡


In [ ]:
# 월세 0원 거래 제외
df = df[df["임대료(만원)"] > 0].copy()
print(f"월세 0원 제거 후: {len(df):,}건")

월세 0원 제거 후: 1,486,299건


In [ ]:
# ㎡당 월세 계산
df["월세_㎡당"] = (
    df["임대료(만원)"] / df["임대면적"]
)

print(f"㎡당 월세 중앙값: {df['월세_㎡당'].median():.2f}만원")
print(f"㎡당 월세 최대값: {df['월세_㎡당'].max():.2f}만원")

㎡당 월세 중앙값: 1.53만원
㎡당 월세 최대값: 521.46만원


In [ ]:
# ㎡당 월세 극단값 처리
# 99.9% 분위수 계산
upper_limit = df["월세_㎡당"].quantile(0.999)

print(f"99.9% 기준값: {upper_limit:.2f}만원/㎡")

# 극단값 제거 전 행 수
before_count = len(df)

# 99.9% 초과 제거
df = df[df["월세_㎡당"] <= upper_limit].copy()

# 제거된 행 수
removed_count = before_count - len(df)

print(f"극단값 제거 건수: {removed_count:,}건")
print(f"극단값 제거 후: {len(df):,}건")
print(f"㎡당 월세 최대값: {df['월세_㎡당'].max():.2f}만원/㎡")

99.9% 기준값: 8.57만원/㎡
극단값 제거 건수: 1,487건
극단값 제거 후: 1,484,812건
㎡당 월세 최대값: 8.57만원/㎡


In [ ]:
print("\n===== 최종 데이터 검증 =====")
print(f"최종 관측치: {len(df):,}건")
print(f"계약일 결측: {df['계약일'].isna().sum():,}건")
print(f"10㎡ 미만: {(df['임대면적'] < 10).sum():,}건")
print(f"월세 0원: {(df['임대료(만원)'] <= 0).sum():,}건")
print(f"㎡당 월세 결측: {df['월세_㎡당'].isna().sum():,}건")
print(f"㎡당 월세 최대값: {df['월세_㎡당'].max():.2f}만원/㎡")


===== 최종 데이터 검증 =====
최종 관측치: 1,484,812건
계약일 결측: 0건
10㎡ 미만: 0건
월세 0원: 0건
㎡당 월세 결측: 0건
㎡당 월세 최대값: 8.57만원/㎡


In [ ]:
df.columns

Index(['접수년도', '자치구코드', '자치구명', '법정동코드', '법정동명', '지번구분코드', '지번구분', '본번', '부번',
       '층', '계약일', '전월세구분', '임대면적', '보증금(만원)', '임대료(만원)', '건물명', '건축년도',
       '건물용도', '계약기간', '신규계약구분', '갱신청구권사용', '종전보증금', '종전임대료', 'address',
       '월세_㎡당'],
      dtype='object')

In [ ]:
# 월별 변수 생성
df["년월"] = (
    df["계약일"].astype(str).str[:4] + "-" +
    df["계약일"].astype(str).str[4:6]
)

monthly_stats = (
    df
    .groupby("년월")
    .agg(
        거래건수=("임대료(만원)", "size"),

        월세_평균=("임대료(만원)", "mean"),
        월세_중앙값=("임대료(만원)", "median"),

        제곱미터당월세_평균=("월세_㎡당", "mean"),
        제곱미터당월세_중앙값=("월세_㎡당", "median"),

        보증금_평균=("보증금(만원)", "mean"),
        보증금_중앙값=("보증금(만원)", "median"),

        면적_평균=("임대면적", "mean"),
        면적_중앙값=("임대면적", "median")
    )
    .reset_index()
    .sort_values("년월")
)

print("\n===== 월별 데이터 =====")
print(monthly_stats.head())

print(f"\n월별 데이터 행 수: {len(monthly_stats):,}개월")
print(f"최종 원자료 행 수: {len(df):,}건")


===== 월별 데이터 =====
        년월  거래건수      월세_평균  월세_중앙값  제곱미터당월세_평균  제곱미터당월세_중앙값       보증금_평균  \
0  2011-01    12  63.750000    50.0    1.181419     1.257625  5625.000000   
1  2011-02    20  50.200000    50.0    1.032475     0.853366  3380.000000   
2  2011-03    17  70.117647    52.0    1.281151     1.299220  3205.882353   
3  2011-04    30  64.233333    52.5    1.030044     1.009054  4136.666667   
4  2011-05    25  63.680000    60.0    1.173510     1.169078  5002.640000   

   보증금_중앙값      면적_평균  면적_중앙값  
0   3500.0  56.440000  59.820  
1   2000.0  56.145500  52.535  
2   2000.0  51.547059  44.520  
3   2500.0  59.378667  52.515  
4   2000.0  57.586800  57.720  

월별 데이터 행 수: 182개월
최종 원자료 행 수: 1,484,812건


#### 2-4-4-1. 금리 데이터 결합

In [ ]:
# 금리 데이터 불러오기

rate_df = pd.read_csv("/content/gov_interest_rate.CSV", encoding="cp949")

In [ ]:
# 금리 데이터의 연월 형식 변환
# 16-Jan → 2016-01

rate_df["년월"] = pd.to_datetime(
    rate_df["연월"],
    format="%y-%b"
).dt.strftime("%Y-%m")

# 2016-01 ~ 2025-12 기간만 필터링

rate_df = rate_df[
    (rate_df["년월"] >= "2016-01") &
    (rate_df["년월"] <= "2025-12")
].copy()

print(f"금리 데이터 기간: {rate_df['년월'].min()} ~ {rate_df['년월'].max()}")
print(f"금리 데이터 월 수: {rate_df['년월'].nunique():,}개월")

# 월별 데이터에 금리 맵핑

df = df.merge(
    rate_df[["년월", "정부대출금금리"]],
    on="년월",
    how="left"
)

# 2016-01 ~ 2025-12 범위 밖 데이터 및 금리 매핑 실패 데이터 제거
df = df[
    (df["년월"] >= "2016-01") &
    (df["년월"] <= "2025-12") &
    (df["정부대출금금리"].notna())
].copy()

# 결과 확인

print(f"최종 데이터: {len(df):,}건")
print(f"최종 데이터 기간: {df['년월'].min()} ~ {df['년월'].max()}")
print(f"금리 맵핑 실패: {df['정부대출금금리'].isna().sum():,}건")

금리 데이터 기간: 2016-01 ~ 2025-12
금리 데이터 월 수: 120개월
최종 데이터: 1,468,362건
최종 데이터 기간: 2016-01 ~ 2025-12
금리 맵핑 실패: 0건


### 2-4-5. 대시보드 제작을 위한 데이터 정제

In [ ]:
# 10㎡ 미만 거래 제외
df = df[df["임대면적"] >= 10].copy()

print(f"10㎡ 미만 제거 후: {len(df):,}건")
print(f"임대면적 최소값: {df['임대면적'].min():.2f}㎡")

10㎡ 미만 제거 후: 246,094건
임대면적 최소값: 10.00㎡


In [ ]:
# 월세 0원 거래 제외
df = df[df["임대료(만원)"] > 0].copy()
print(f"월세 0원 제거 후: {len(df):,}건")

월세 0원 제거 후: 246,069건


In [ ]:
# ㎡당 월세 계산
df["월세_㎡당"] = (
    df["임대료(만원)"] / df["임대면적"]
)

print(f"㎡당 월세 중앙값: {df['월세_㎡당'].median():.2f}만원")
print(f"㎡당 월세 최대값: {df['월세_㎡당'].max():.2f}만원")

㎡당 월세 중앙값: 1.81만원
㎡당 월세 최대값: 40.30만원


In [ ]:
# ㎡당 월세 극단값 처리
# 99.9% 분위수 계산
upper_limit = df["월세_㎡당"].quantile(0.999)

print(f"99.9% 기준값: {upper_limit:.2f}만원/㎡")

# 극단값 제거 전 행 수
before_count = len(df)

# 99.9% 초과 제거
df = df[df["월세_㎡당"] <= upper_limit].copy()

# 제거된 행 수
removed_count = before_count - len(df)

print(f"극단값 제거 건수: {removed_count:,}건")
print(f"극단값 제거 후: {len(df):,}건")
print(f"㎡당 월세 최대값: {df['월세_㎡당'].max():.2f}만원/㎡")

99.9% 기준값: 10.02만원/㎡
극단값 제거 건수: 247건
극단값 제거 후: 245,822건
㎡당 월세 최대값: 10.01만원/㎡


In [ ]:
print("\n===== 최종 데이터 검증 =====")
print(f"최종 관측치: {len(df):,}건")
print(f"계약일 결측: {df['계약일'].isna().sum():,}건")
print(f"10㎡ 미만: {(df['임대면적'] < 10).sum():,}건")
print(f"월세 0원: {(df['임대료(만원)'] <= 0).sum():,}건")
print(f"㎡당 월세 결측: {df['월세_㎡당'].isna().sum():,}건")
print(f"㎡당 월세 최대값: {df['월세_㎡당'].max():.2f}만원/㎡")


===== 최종 데이터 검증 =====
최종 관측치: 245,822건
계약일 결측: 0건
10㎡ 미만: 0건
월세 0원: 0건
㎡당 월세 결측: 0건
㎡당 월세 최대값: 10.01만원/㎡


In [ ]:
df.columns

Index(['접수년도', '자치구코드', '자치구명', '법정동코드', '법정동명', '지번구분코드', '지번구분', '본번', '부번',
       '층', '계약일', '전월세구분', '임대면적', '보증금(만원)', '임대료(만원)', '건물명', '건축년도',
       '건물용도', '계약기간', '신규계약구분', '갱신청구권사용', '종전보증금', '종전임대료', 'address', '위도',
       '경도', '월세_㎡당'],
      dtype='object')

In [ ]:
# 월별 변수 생성
df["년월"] = (
    df["계약일"].astype(str).str[:4] + "-" +
    df["계약일"].astype(str).str[4:6]
)

monthly_stats = (
    df
    .groupby("년월")
    .agg(
        거래건수=("임대료(만원)", "size"),

        월세_평균=("임대료(만원)", "mean"),
        월세_중앙값=("임대료(만원)", "median"),

        제곱미터당월세_평균=("월세_㎡당", "mean"),
        제곱미터당월세_중앙값=("월세_㎡당", "median"),

        보증금_평균=("보증금(만원)", "mean"),
        보증금_중앙값=("보증금(만원)", "median"),

        면적_평균=("임대면적", "mean"),
        면적_중앙값=("임대면적", "median")
    )
    .reset_index()
    .sort_values("년월")
)

print("\n===== 월별 데이터 =====")
print(monthly_stats.head())

print(f"\n월별 데이터 행 수: {len(monthly_stats):,}개월")
print(f"최종 원자료 행 수: {len(df):,}건")


===== 월별 데이터 =====
        년월   거래건수      월세_평균  월세_중앙값  제곱미터당월세_평균  제곱미터당월세_중앙값        보증금_평균  \
0  2025-01  20777  80.198729    61.0    2.127369     1.882506  12830.294942   
1  2025-02  24913  78.447678    60.0    2.039777     1.795332  12626.281821   
2  2025-03  20944  82.115021    60.0    1.958135     1.706795  13603.912672   
3  2025-04  20198  81.559956    60.0    1.916879     1.667626  13010.989108   
4  2025-05  21449  82.019675    60.0    1.973602     1.729615  13109.838547   

   보증금_중앙값      면적_평균  면적_중앙값  
0   5000.0  42.140464   31.33  
1   5238.0  43.186100   34.39  
2   6700.0  45.750582   37.96  
3   6000.0  45.915022   39.02  
4   6000.0  45.164493   38.00  

월별 데이터 행 수: 12개월
최종 원자료 행 수: 245,822건


#### 2-4-5-1. 금리 데이터 결합

In [ ]:
# 금리 데이터 불러오기
rate_df = pd.read_csv(
    "/content/gov_interest_rate.CSV",
    encoding="cp949"
)

# 금리 데이터의 연월 형식 변환
rate_df["년월"] = pd.to_datetime(
    rate_df["연월"],
    format="%y-%b"
).dt.strftime("%Y-%m")

# 월별 데이터에 금리 맵핑
monthly_stats = monthly_stats.merge(
    rate_df[["년월", "정부대출금금리"]],
    on="년월",
    how="left"
)

# 확인
print(monthly_stats.head())

print(f"전체 월 수: {len(monthly_stats):,}")
print(f"금리 맵핑 성공: {monthly_stats['정부대출금금리'].notna().sum():,}")
print(f"금리 맵핑 실패: {monthly_stats['정부대출금금리'].isna().sum():,}")

        년월   거래건수      월세_평균  월세_중앙값  제곱미터당월세_평균  제곱미터당월세_중앙값        보증금_평균  \
0  2025-01  20777  80.198729    61.0    2.127369     1.882506  12830.294942   
1  2025-02  24913  78.447678    60.0    2.039777     1.795332  12626.281821   
2  2025-03  20944  82.115021    60.0    1.958135     1.706795  13603.912672   
3  2025-04  20198  81.559956    60.0    1.916879     1.667626  13010.989108   
4  2025-05  21449  82.019675    60.0    1.973602     1.729615  13109.838547   

   보증금_중앙값      면적_평균  면적_중앙값  정부대출금금리  
0   5000.0  42.140464   31.33    3.069  
1   5238.0  43.186100   34.39    3.069  
2   6700.0  45.750582   37.96    3.069  
3   6000.0  45.915022   39.02    2.817  
4   6000.0  45.164493   38.00    2.817  
전체 월 수: 12
금리 맵핑 성공: 12
금리 맵핑 실패: 0


In [ ]:
# 저장
OUTPUT_PATH = "/content/main_processed_data.csv"

df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print(f"저장 완료: {OUTPUT_PATH}")
print(f"데이터 크기: {df.shape}")

저장 완료: /content/main_processed_data.csv
데이터 크기: (1468362, 27)


In [ ]:
df.to_parquet(
    "/content/main_processed_data.parquet",
    index=False
)

print("저장 완료!")

저장 완료!


### 2-4-5-2. 최근접 지하철역 정보 결합

In [ ]:
# 데이터 불러오기
df = pd.read_csv("/content/rental_filtered_2025_coordinated_interestrated.csv", encoding="utf-8-sig")
subway_df = pd.read_csv("/content/지하철역_GEOM (역사마스터).csv",encoding="utf-8-sig")

In [ ]:
from sklearn.neighbors import BallTree

# 위도/경도 결측치 제거 대상 확인
print("전체 데이터:", len(df))
print("위도 결측:", df["위도"].isna().sum())
print("경도 결측:", df["경도"].isna().sum())

In [ ]:
# BallTree 생성
# 위도/경도를 라디안으로 변환
subway_coords = np.radians(
    subway_df[["위도", "경도"]].values
)

tree = BallTree(
    subway_coords,
    metric="haversine"
)


In [ ]:
# 월세 데이터 중 좌표가 있는 데이터만 계산
valid_mask = df["위도"].notna() & df["경도"].notna()

df_valid = df.loc[valid_mask].copy()

df_coords = np.radians(
    df_valid[["위도", "경도"]].values
)

In [ ]:
# 가장 가까운 지하철역 찾기
distances, indices = tree.query(
    df_coords,
    k=1
)

# 거리 변환
# 지구 반지름 약 6,371,000m
distances_m = distances[:, 0] * 6_371_000

nearest_indices = indices[:, 0]

# 최근접역 정보 가져오기
nearest_subway = subway_df.iloc[
    nearest_indices
].reset_index(drop=True)

In [ ]:
# 컬럼 생성
df.loc[valid_mask, "최근접역"] = (nearest_subway["역명"].values)

df.loc[valid_mask, "최근접역_호선"] = (nearest_subway["호선"].values)

df.loc[valid_mask, "최근접역_거리(m)"] = (distances_m)

df.loc[valid_mask, "최근접역_위도"] = (nearest_subway["위도"].values)

df.loc[valid_mask, "최근접역_경도"] = (nearest_subway["경도"].values)

In [ ]:
# 거리 구간 생성
df["거리구간"] = pd.cut(
    df["최근접역_거리(m)"],
    bins=[
        -np.inf,
        250,
        500,
        1000,
        1500,
        np.inf
    ],
    labels=[
        "250m 이하",
        "250m 초과 ~ 500m 이하",
        "500m 초과 ~ 1km 이하",
        "1km 초과 ~ 1.5km 이하",
        "1.5km 초과"
    ]
)

In [ ]:
# 결과 확인
print(df[
    [
        "위도",
        "경도",
        "최근접역",
        "최근접역_호선",
        "최근접역_거리(m)",
        "거리구간",
        "최근접역_위도",
        "최근접역_경도"
    ]
].head())

In [ ]:
# 저장
df.to_csv(
    "filtered_data_for_dashboard.csv",
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료")

In [ ]:
# GitHub 업로드용 parquet 데이터 저장
df = pd.read_csv('/content/filtered_data_for_dashboard.csv', encoding="utf-8-sig")

df.to_parquet(
    "/content/df_2025.parquet",
    index=False
)

print("저장 완료!")

저장 완료!


In [ ]:
## 3-4. 금리 변수와 월세 가격의 상관 관계 비교
월세 시장과 가장 밀접한 금리 변수를 선정하기 위해
세 가지 금리 변수와 월세 가격 변수 간의 상관관계를 비교한다.
### 3-4-1. 단순 비교
월세 시장과 가장 밀접한 금리 변수를 선정하기 위해
세 가지 금리 변수와 월세 가격 변수 간의 상관관계를 비교한다.

분석 대상 금리:
- 기준금리
- 정부대출금리
- 국고채10년

분석 대상 월세 가격:
- 평당 임대료
- 평당 보증금

상관관계는 두 가지 방법으로 측정한다.

1. Pearson 상관계수
   - 두 변수 간의 선형 관계를 측정
   - 상관계수의 범위는 -1 ~ +1
   - 절댓값이 클수록 선형 관계가 강함

2. Spearman 상관계수
   - 두 변수의 순위를 이용하여 단조 관계를 측정
   - 가격 데이터의 비정규성이나 이상치에 상대적으로 강건
   - 이번 분석에서는 Spearman을 주요 판단 기준으로 사용한다.

또한 각 상관계수의 통계적 유의성을 확인하기 위해 p-value를 함께 확인한다.
분석 변수 설정
import pandas as pd
import numpy as np
from scipy.stats import pearsonr, spearmanr

# 분석 대상 변수 설정
rate_cols = [
    '기준금리',
    '정부대출금리',
    '국고채10년'
]

price_cols = [
    '평당 임대료(만원/평)',
    '평당 보증금(만원/평)'
]

# 분석에 필요한 변수만 추출
corr_df = df_monthly[rate_cols + price_cols].copy()
금리별 결측치 확인
corr_df.isnull().sum()
# 결측치
corr_df = corr_df.dropna(subset=['평당 임대료(만원/평)']).copy()
corr_df.isnull().sum()
Pearson + Spearman 상관계수 계산

세 가지 금리와 두 가지 월세 가격 변수의 관계를 각각 계산한다.

- Pearson r: 선형 상관관계
- Spearman ρ: 순위 기반 단조 상관관계
- p-value: 해당 상관관계가 0이라는 귀무가설 검정

표본 수가 매우 크기 때문에 p-value 자체보다는
**상관계수의 크기와 방향**을 주요 판단 기준으로 사용한다.
results = []

for rate in rate_cols:
    for price in price_cols:
        pearson_r, pearson_p = pearsonr(corr_df[rate], corr_df[price])
        spearman_r, spearman_p = spearmanr(corr_df[rate], corr_df[price])

        results.append({
            '금리': rate,
            '가격변수': price,
            '표본수': len(corr_df),
            'Pearson_r': round(pearson_r, 6),
            'Pearson_p': round(pearson_p, 6),
            'Spearman_rho': round(spearman_r, 6),
            'Spearman_p': round(spearman_p, 6)
        })

corr_results = pd.DataFrame(results)
corr_results
상관 관계가 없음

### 3-4-2. 월세 가격 변동 및 금리변동 파생변수 생성
 **금리가 상승하거나 하락했을 때 월세 가격이 실제로 어떻게 움직이는지**를 확인
 금리의 절대값 대신 **전월 대비 금리의 변동 방향**을 나타내는 파생변수를 생성


- 금리 상승 → `1`
- 금리 동결 → `0`
- 금리 하락 → `-1`

월세 가격의 경우 서울 전체의 평균 가격을 기준으로 비교하지 않고,
각 거래의 `최근접역`을 기준으로 전월 가격과 비교한다.

예를 들어,

> 2024년 5월 A역 인근 거래의 평당 임대료가 60만원이고
> 2024년 4월 A역 인근 거래의 평당 임대료 중앙값이 55만원이라면

- 전월대비 평당 임대료 변동금액 = `+5만원/평`
- 전월대비 평당 임대료 변동률 = `+9.09%`

로 계산한다.

이를 통해 지역별 월세 수준의 차이를 고려하면서,
**금리변동과 역세권별 월세 가격변동의 관계**를 분석한다.

분석 원칙

1. 금리변동은 `계약연월` 기준으로 전월과 비교한다.
2. 월세 가격의 기준값은 `최근접역 + 계약연월`별 중앙값을 사용한다.
3. 개별 거래는 해당 역의 **전월 중앙값**과 비교한다.
4. 전월 거래가 존재하지 않는 경우 변동값을 계산하지 않는다.
5. 첫 번째 월의 금리변동 역시 전월 데이터가 없으므로 결측치로 처리한다.
# 계약월 변수 정리
df_monthly['계약연월'] = pd.to_datetime(df_monthly['계약연월'])
df_monthly['계약연월'].head()
# 역세권별 월별 대표가격 산출
# `최근접역`과 `계약연월`을 기준으로 월별 평당 임대료와 평당 보증금의 중앙값을 계산한다.

station_month = (
    df_monthly
    .groupby(['최근접역', '계약연월'], observed=True)[
        ['평당 임대료(만원/평)', '평당 보증금(만원/평)']
    ]
    .median()
    .reset_index()
)
# 전월 기준가격 생성
station_month['계약연월'] = station_month['계약연월'].dt.to_period('M')

df_monthly['계약연월'] = df_monthly['계약연월'].dt.to_period('M')

df_monthly['전월'] = df_monthly['계약연월'] - 1
station_month = station_month.rename(columns={
    '계약연월': '전월',
    '평당 임대료(만원/평)': '전월_역세권_평당임대료_중앙값',
    '평당 보증금(만원/평)': '전월_역세권_평당보증금_중앙값'
})

df_monthly = df_monthly.merge(
    station_month,
    on=['최근접역', '전월'],
    how='left'
)
# 월세 변동금액·변동률 생성
# 현재 거래가격을 동일 역세권의 전월 중앙값과 비교한다.

# 변동금액은 실제 가격이 얼마나 차이 나는지를 나타내고,
# 변동률은 전월 가격 대비 상대적으로 얼마나 차이가 나는지를 나타낸다.

df_monthly['전월대비_평당임대료_변동금액'] = (
    df_monthly['평당 임대료(만원/평)']
    - df_monthly['전월_역세권_평당임대료_중앙값']
)

df_monthly['전월대비_평당임대료_변동률'] = (
    df_monthly['전월대비_평당임대료_변동금액']
    / df_monthly['전월_역세권_평당임대료_중앙값']
    * 100
)

df_monthly['전월대비_평당보증금_변동금액'] = (
    df_monthly['평당 보증금(만원/평)']
    - df_monthly['전월_역세권_평당보증금_중앙값']
)

df_monthly['전월대비_평당보증금_변동률'] = (
    df_monthly['전월대비_평당보증금_변동금액']
    / df_monthly['전월_역세권_평당보증금_중앙값']
    * 100
)
# 금리 변동 생성
# 각 계약연월의 금리를 전월과 비교하여 변동 방향을 나타내는 변수를 생성한다.
rate_month = (
    df_monthly
    .groupby('계약연월', observed=True)[
        ['기준금리', '정부대출금리', '국고채10년']
    ]
    .first()
    .sort_index()
)

for col in ['기준금리', '정부대출금리', '국고채10년']:
    rate_month[f'{col}_변동'] = np.sign(rate_month[col].diff()).astype('Int8')

df_monthly = df_monthly.merge(
    rate_month[
        ['기준금리_변동', '정부대출금리_변동', '국고채10년_변동']
    ],
    left_on='계약연월',
    right_index=True,
    how='left'
)
# 확인
df_monthly[[
    '계약연월',
    '최근접역',
    '평당 임대료(만원/평)',
    '전월_역세권_평당임대료_중앙값',
    '전월대비_평당임대료_변동금액',
    '전월대비_평당임대료_변동률',
    '기준금리',
    '기준금리_변동'
]].head(10)
df_monthly[
    ['기준금리_변동', '정부대출금리_변동', '국고채10년_변동']
].apply(lambda x: x.value_counts(dropna=False))
# 전월 가격 기준값이 없는 거래 확인
df_monthly[
    '전월_역세권_평당임대료_중앙값'
].isna().mean() * 100
df_monthly[
    '전월_역세권_평당임대료_중앙값'
].isna().sum()
# 2016년 2월 이후, 월세 가격변동이 계산 가능한 거래만 분석
analysis_df = df_monthly[
    (df_monthly['계약연월'] >= pd.Period('2016-02', freq='M')) &
    (df_monthly['전월_역세권_평당임대료_중앙값'].notna()) &
    (df_monthly['전월_역세권_평당보증금_중앙값'].notna())
].copy()

print(f'분석 대상 거래: {len(analysis_df):,}건')
analysis_df.groupby('기준금리_변동')[
    ['전월대비_평당임대료_변동금액',
     '전월대비_평당임대료_변동률',
     '전월대비_평당보증금_변동금액',
     '전월대비_평당보증금_변동률']
].agg(['count', 'mean', 'median', 'std']).round(3)
for rate in ['기준금리_변동', '정부대출금리_변동', '국고채10년_변동']:
    print(f'\n===== {rate} =====')

    print(
        analysis_df.groupby(rate)[
            ['전월대비_평당임대료_변동률',
             '전월대비_평당보증금_변동률']
        ]
        .agg(['count', 'mean', 'median'])
        .round(3)
    )
금리변동이 상승·동결·하락일 때 월세 가격변동의 분포가 통계적으로 차이가 있는지 검정한다.

거래가격의 분포가 정규분포라고 가정하기 어렵고 이상치의 영향을 줄이기 위해
비모수 검정인 Kruskal-Wallis 검정을 사용한다.

귀무가설(H₀):
금리변동 방향에 따른 월세 가격변동 분포의 차이가 없다.

대립가설(H₁):
적어도 하나의 금리변동 그룹에서 월세 가격변동 분포가 다르다.
# 금리변동 그룹 간 월세 가격변화 차이 검정
from scipy.stats import kruskal

for rate in ['기준금리_변동', '정부대출금리_변동', '국고채10년_변동']:
    print(f'\n===== {rate} =====')

    groups = [
        analysis_df.loc[
            analysis_df[rate] == change,
            '전월대비_평당임대료_변동률'
        ].dropna()
        for change in [-1, 0, 1]
    ]

    stat, p = kruskal(*groups)

    print(f'Kruskal-Wallis 통계량: {stat:.3f}')
    print(f'p-value: {p:.6g}')